# Verify: violation was genuinely costless in the partial arm

Confirms the 2x2 is clean: in the **partial (violator-cost-only) arm** `extinct3.csv`, the
25-step timeout removal was disabled at the switch together with the zap penalty, so a
zapped violator loses **nothing** in the ghost phase.

No GPU, no retraining, **no third-party packages** (stdlib `csv` only). Reads CSVs already on
disk. Run all cells; the last one prints PASS/FAIL.

**Why two files:** `extinct3.csv` was generated before the `removal_on` column existed, so it
logs `enforce` (which the removal was welded to) but not `removal_on`. The tiny
`extinct3_removal_smoke.csv` was generated with current code and carries `removal_on`, so it
demonstrates the gate machinery directly.

In [ ]:
import csv

def load(path):
    with open(path, newline='') as f:
        return list(csv.DictReader(f))

N_INSTALL = 1000        # enforce ON for update < 1000, ghost after
ok = True

# --- (A) Partial arm: enforce (penalty channel) drops to 0 at the switch -------------
d = load('extinct3.csv')
assert 'removal_on' not in d[0], 'unexpected: this file already has removal_on'
pre  = [r for r in d if int(r['update']) <  N_INSTALL]
post = [r for r in d if int(r['update']) >= N_INSTALL]
a_pre  = all(float(r['enforce']) == 1.0 for r in pre)
a_post = all(float(r['enforce']) == 0.0 for r in post)
print('extinct3.csv (partial arm):')
print(f'  enforce, updates <{N_INSTALL}: all == 1.0  -> {a_pre}')
print(f'  enforce, updates >={N_INSTALL}: all == 0.0  -> {a_post}')
print('  (removal_on column absent -> file predates the independent gate; removal was')
print('   welded to enforce, so enforce==0 after switch == timeout==0 after switch)')
ok = ok and a_pre and a_post

In [ ]:
# --- (B) Machinery on current code: removal_on gates independently, and IS logged ------
s = load('extinct3_removal_smoke.csv')
assert 'enforce' in s[0] and 'removal_on' in s[0]
GATE = 10               # smoke used gate_removal_after=10, enforce never gated
smin = min(int(r['seed']) for r in s)
s0 = [r for r in s if int(r['seed']) == smin]
enf_on   = all(float(r['enforce'])    == 1.0 for r in s0)                          # penalty kept
rem_pre  = all(float(r['removal_on']) == 1.0 for r in s0 if int(r['update']) <  GATE)
rem_post = all(float(r['removal_on']) == 0.0 for r in s0 if int(r['update']) >= GATE)
print('extinct3_removal_smoke.csv (current code, gate_removal_after=10):')
print(f'  enforce == 1.0 throughout (penalty kept)     -> {enf_on}')
print(f'  removal_on == 1.0 for update <{GATE}          -> {rem_pre}')
print(f'  removal_on == 0.0 for update >={GATE}          -> {rem_post}')
print('  => timeout gates independently of the penalty, and is logged per update')
ok = ok and enf_on and rem_pre and rem_post

In [ ]:
# --- Verdict --------------------------------------------------------------------------
print('=' * 68)
if ok:
    print('PASS - the 2x2 is clean.')
    print('  Partial arm: enforce (zap penalty) OFF after update 1000, and the')
    print('  25-step timeout was welded to enforce (default gate_removal_after=None,')
    print('  train_jax.py:249 enf_removal_sched = enforce_sched), so the timeout was')
    print('  OFF too. A zapped violator bore no penalty and no removal in the ghost')
    print('  phase, yet silly avoidance HELD -> the norm rests on the enforcer\'s')
    print('  incentive, not the violator\'s cost.')
else:
    print('FAIL - inspect the printouts above.')
print('=' * 68)